In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_path = '/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_im_3/train_us.csv'
df = pd.read_csv(train_path)
df

,user_id,course_id,course_duration,start_year,start_month,start_day,enrollment_to_end,label_3,gender,num_course_order,...,cmt_p4_max_comment_length_phase4,cmt_p4_text_diversity_phase4,cmt_p4_comment_days_active_phase4,cmt_p4_entropy_time_mean_phase4,cmt_p4_most_common_time_bin_phase4,cmt_p4_time_entropy_var_phase4,cmt_p4_positive_ratio_phase4,cmt_p4_neutral_ratio_phase4,cmt_p4_negative_ratio_phase4,cmt_p4_sentiment_entropy_phase4
0,499264,633,250,2019,12,25,194,2,1.0,3,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
1,112819,869,121,2020,4,1,112,2,0.0,51,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
2,606920,714,121,2020,9,1,100,2,1.0,4,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
3,556258,792,173,2020,2,9,150,2,1.0,20,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
4,846799,833,93,2020,4,13,46,2,2.0,1,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206617,178425,847,104,2020,4,13,28,2,2.0,1,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
206618,1096149,777,137,2020,2,14,64,2,2.0,9,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
206619,429165,814,169,2020,2,13,146,2,1.0,19,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0
206620,1124303,845,77,2019,12,29,75,2,1.0,1,...,8.0,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0


In [ ]:
!pip install imblearn

In [ ]:
def radius_smote_generate(X, y, minority_label, target_count, k_neighbors=5):
    X = np.asarray(X)
    y = np.asarray(y)

    X_min = X[y == minority_label]
    X_maj = X[y != minority_label]

    need = target_count - len(X_min)
    if need <= 0:
        print(f"Lớp {minority_label} đã đủ {target_count} mẫu.")
        return X, y

    # --- Filter SAFE using KNN ---
    knn = KNeighborsClassifier(n_neighbors=k_neighbors)
    knn.fit(X, y)
    pred = knn.predict(X_min)

    X_safe = X_min[pred == minority_label]
    if len(X_safe) == 0:
        print("Không có SAFE sample.")
        return X, y

    # số synthetic mỗi SAFE cần sinh
    per_sample = int(np.ceil(need / len(X_safe)))

    synthetic = []

    for p in X_safe:
        # nearest majority
        dist = np.sqrt(np.sum((X_maj - p)**2, axis=1))
        t = X_maj[np.argmin(dist)]
        r_vec = t - p
        r = np.linalg.norm(r_vec)

        for _ in range(per_sample):
            alpha = np.random.rand()

            a = p + alpha * r_vec
            b = p + alpha * (-r_vec)

            if np.linalg.norm(a - p) <= r:
                synthetic.append(a)
            if np.linalg.norm(b - p) <= r:
                synthetic.append(b)

            if len(synthetic) >= need:
                break
        if len(synthetic) >= need:
            break

    synthetic = np.array(synthetic[:need])

    # merge
    X_new = np.vstack([X, synthetic])
    y_new = np.hstack([y, np.full(len(synthetic), minority_label)])

    print(f"Tạo xong {len(synthetic)} mẫu cho lớp {minority_label}.")
    return X_new, y_new

In [ ]:
X = df.drop(columns=["label_3"])
y = df["label_3"]

In [ ]:
TARGET = 200000
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Tăng nhãn 1 lên 200k
X1, y1 = radius_smote_generate(X, y, minority_label=1, target_count=TARGET)

# Tăng nhãn 0 lên 200k
X2, y2 = radius_smote_generate(X1, y1, minority_label=0, target_count=TARGET)


Tạo xong 195161 mẫu cho lớp 1.
Tạo xong 198217 mẫu cho lớp 0.


In [ ]:
import pandas as pd
pd.Series(y2).value_counts()

,count
2,200000
1,200000
0,200000


In [ ]:
df_new = pd.concat([pd.DataFrame(X2, columns=df.drop(columns=["label_3"]).columns), pd.Series(y2, name='label_3')], axis=1)
df_new

,user_id,course_id,course_duration,start_year,start_month,start_day,enrollment_to_end,gender,num_course_order,avg_interval_enroll_time,...,cmt_p4_text_diversity_phase4,cmt_p4_comment_days_active_phase4,cmt_p4_entropy_time_mean_phase4,cmt_p4_most_common_time_bin_phase4,cmt_p4_time_entropy_var_phase4,cmt_p4_positive_ratio_phase4,cmt_p4_neutral_ratio_phase4,cmt_p4_negative_ratio_phase4,cmt_p4_sentiment_entropy_phase4,label_3
0,499264.000000,633.0,250.0,2019.0,12.0,25.0,194.000000,1.0,3.0,0.035000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,2
1,112819.000000,869.0,121.0,2020.0,4.0,1.0,112.000000,0.0,51.0,8.034600,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,2
2,606920.000000,714.0,121.0,2020.0,9.0,1.0,100.000000,1.0,4.0,68.370000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,2
3,556258.000000,792.0,173.0,2020.0,2.0,9.0,150.000000,1.0,20.0,4.303158,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,2
4,846799.000000,833.0,93.0,2020.0,4.0,13.0,46.000000,2.0,1.0,0.000000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,417106.499462,769.0,139.0,2020.0,2.0,17.0,59.361387,2.0,1.0,0.000000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,0
599996,417065.500538,769.0,139.0,2020.0,2.0,17.0,60.638613,2.0,1.0,0.000000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,0
599997,417205.259684,769.0,139.0,2020.0,2.0,17.0,56.284745,2.0,1.0,0.000000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,0
599998,416966.740316,769.0,139.0,2020.0,2.0,17.0,63.715255,2.0,1.0,0.000000,...,33.0,1.0,0.615991,31.0,0.0,0.0,0.818182,0.0,0.0,0


In [ ]:
df_new.to_csv('/content/drive/MyDrive/Nhóm 1/Dataset/Data_cleaned/split_data/data_median_radiussmote/train.csv')

In [ ]:
dis